<a href="https://colab.research.google.com/github/varakalicheva-hub/compling/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82_3_%D0%9A%D0%B0%D0%BB%D0%B8%D1%87%D0%B5%D0%B2%D0%B0_%D0%9C%D0%B0%D1%80%D0%BA%D0%BE%D0%B2%D0%B8%D1%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Работа с векторной БД ChromaDB и кастомным RAG-пайплайном**

# Набор документов

In [ ]:
# выполняем вход, чтобы загружать закрытые модели и датасеты
from google.colab import userdata

HF_TOKEN = userdata.get('HF_WRITE_TOKEN')

from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
!pip install langchain langchain_text_splitters -q
!pip install llama-index-core llama-index-embeddings-huggingface llama-index-llms-openrouter -q
!pip install langchain-openai langgraph -q
!pip install datasets pandas -q
!pip install llama-index-llms-langchain -q
!pip install langchain-core langchain-community -q

print("Установка завершена")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 597.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take i

In [ ]:
!pip install chromadb sentence-transformers tensorflow-datasets langchain-huggingface -q langchain_community

In [ ]:
import tensorflow_datasets as tfds
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSpliяtter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import warnings
warnings.filterwarnings('ignore')

print("Все библиотеки импортированы успешно")

Все библиотеки импортированы успешно


In [ ]:
import tensorflow_datasets as tfds

# загрузка 100 текстов отзывов из датасета IMDB Reviews через библиотеку tensorflow_datasets
def load_imdb_texts(n=1000):
    ds = tfds.load("imdb_reviews", split=f"train[:{n}]", as_supervised=True) # берем примеры из тренировочной части
    texts = []
    for text, _ in ds:  # _ игнорируем метку (тональность), так как нам нужен только текст
        texts.append(text.numpy().decode('utf-8'))
    return texts

texts = load_imdb_texts(1000)
print(f"Загружено {len(texts)} текстов")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.F059NV_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.F059NV_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.F059NV_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
Загружено 1000 текстов


In [ ]:
documents = []

for i, text in enumerate(texts): # перебираем тексты отзывов, чтобы получить индекс (i) и сам текст (text).
    doc = Document( # создаем объект, в котором будет хранится текст и метаданные
        page_content=text,
        metadata={"id": i, "source": "reviews_films_imbd"}
    )
    documents.append(doc)
# проверяем
print(f"Создано {len(documents)} документов")
print(f"Пример: {documents[0].page_content[:500]}...")
print(f"Метаданные: {documents[0].metadata}")

Создано 1000 документов
Пример: This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but...
Метаданные: {'id': 0, 'source': 'reviews_films_imbd'}


# Статистические метрики датасета

In [ ]:
import numpy as np
import re

def split_into_sentences(text):
    """
    Разбивает текст на предложения по знакам . ! ?
    """
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

# Собираем статистику
text_lengths = []
sentence_counts = []
sentence_lengths = []

for doc in documents:
    text = doc.page_content
    sentences = split_into_sentences(text)
    text_lengths.append(len(text))
    sentence_counts.append(len(sentences))
    for sent in sentences:
        sentence_lengths.append(len(sent))

# Вычисляем квартили
q1_len = np.percentile(text_lengths, 25)
q3_len = np.percentile(text_lengths, 75)

# Считаем короткие (≤ Q1) и длинные (≥ Q3) отзывы
short_reviews = [l for l in text_lengths if l <= q1_len]
long_reviews = [l for l in text_lengths if l >= q3_len]
num_short = len(short_reviews)
num_long = len(long_reviews)
percent_short = num_short / len(text_lengths) * 100
percent_long = num_long / len(text_lengths) * 100

print("\nДлина текстов по символам:")
print(f"  Среднее: {np.mean(text_lengths):.0f}")
print(f"  Медиана: {np.median(text_lengths):.0f}")
print(f"  Минимум: {min(text_lengths)}")
print(f"  Максимум: {max(text_lengths)}")
print(f"  25-й перцентиль (Q1): {q1_len:.0f}")
print(f"  75-й перцентиль (Q3): {q3_len:.0f}")

print("\nКоличество предложений в отзыве:")
print(f"  Среднее: {np.mean(sentence_counts):.1f}")
print(f"  Медиана: {np.median(sentence_counts):.0f}")
print(f"  Минимум: {min(sentence_counts)}")
print(f"  Максимум: {max(sentence_counts)}")

print("\nДлина предложений по символам:")
print(f"  Средняя: {np.mean(sentence_lengths):.1f}")
print(f"  Медиана: {np.median(sentence_lengths):.0f}")
print(f"  Минимум: {min(sentence_lengths)}")
print(f"  Максимум: {max(sentence_lengths)}")

print("\nРаспределение по длине отзывов:")
print(f"  Короткие отзывы (≤ Q1, ≤{q1_len:.0f} символов): {num_short} шт. ({percent_short:.1f}%)")
print(f"  Длинные отзывы (≥ Q3, ≥{q3_len:.0f} символов): {num_long} шт. ({percent_long:.1f}%)")
print(f"  Средние (между Q1 и Q3): {len(text_lengths) - num_short - num_long} шт. ({100 - percent_short - percent_long:.1f}%)")


Длина текстов по символам:
  Среднее: 1279
  Медиана: 978
  Минимум: 123
  Максимум: 5578
  25-й перцентиль (Q1): 689
  75-й перцентиль (Q3): 1610

Количество предложений в отзыве:
  Среднее: 13.2
  Медиана: 11
  Минимум: 2
  Максимум: 145

Длина предложений по символам:
  Средняя: 95.4
  Медиана: 83
  Минимум: 1
  Максимум: 748

Распределение по длине отзывов:
  Короткие отзывы (≤ Q1, ≤689 символов): 252 шт. (25.2%)
  Длинные отзывы (≥ Q3, ≥1610 символов): 250 шт. (25.0%)
  Средние (между Q1 и Q3): 498 шт. (49.8%)


# Разделение на чанки


**Обоснование выбора**

1. В ходе сравнения качества разделителей текста (CharacterTextSplitter и RecursiveCharacterTextSplitter) был выбран последний, несмотря на то, что область применения первого — короткие и неструктурированные тексты, то есть покрывает большую часть наших отзывов (в большинстве своем они короткие). Мы связываем такой результат с тем, что RecursiveCharacterTextSplitter разделяет текст логичнее с точки зрения контекста — отдает предпочтение абзацам и предложениями, а не пробелам, максимальтно осмысленно раздел, не превышая при этом лимиты размера. Создание подобных естественных, контекстно-зависимых фрагментов текста идеально подходят *для* системы поиска, создаение которой и является целью проекта.

2. Размер чанка 2700 символов выбран так, чтобы покрыть 75% отзывов (длиной до 1610) целиком, сохраняя их смысловую целостность, снизив потерю контекста при разбиении и вероятность наличия "оборванных" фраз и слов.  Оставшиеся длинные отзывы (максимум 5578) при необходимости разбиваются.

3. Перекрытие 60 символов (≈2% от чанка) выбрано минимальным, чтобы сохранить связность только в тех редких случаях, когда отзыв длиннее 2700 символов и требует разбиения. Это позволяет не терять контекст на границах, не создавая избыточности для большинства отзывов, которые умещаются в один чанк.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2700,  # размер чанка в символах
    chunk_overlap=60, # перекрытие
    separators=['\n\n', '\n', '.', '!', '?', '...', ' ', ''] # иерархия разделителей
)

# выводим 10 чанков, чтобы убедиться в их качестве
chunked_docs = text_splitter.split_documents(documents)

print(f"\nВсего чанков: {len(chunked_docs)}")
print("\nПервые 10 чанков (полный текст):\n")
for i, doc in enumerate(chunked_docs[:10]):
    print(f"Чанк {i+1}:\n{doc.page_content}\nМетаданные: {doc.metadata}\n{'-'*40}")


Всего чанков: 1079

Первые 10 чанков (полный текст):

Чанк 1:
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.
Метаданные: {'id': 0, 'source': 'reviews_films_imbd'}
----------------------------------------
Чанк 2:
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, b

# Эмбеддинги: загружаем модель

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# выбираем модель
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Сохранение эмбеддингов и метаданных в БД

In [ ]:
# создаем векторное хранилище
vectorstore = Chroma.from_documents(
   documents=chunked_docs,
   embedding=embedding_model,
   persist_directory="./chroma_imdb_db"
)

print("Векторное хранилище Chroma создано")
print(f"Количество векторов: {vectorstore._collection.count()}")

Векторное хранилище Chroma создано
Количество векторов: 1079


In [ ]:
# сохраняем БД
vectorstore.persist()

print("Векторная база данных сохранена в ./chroma_imdb_db")

Векторная база данных сохранена в ./chroma_imdb_db


# Функция семантического поиска

 Был выбран similarity_search_with_score, потому что он возвращает не только найденные фрагменты, но и числовую оценку их близости к запросу. Это позволяет понять, насколько каждый чанк подходит по смыслу и отсеять малополезные куски (с высоким score, так как score — это расстояние и чем оно меньше, тем ближе чанк по смыслу к запросу), передав в языковую модель только самые релевантные и отсортированные по степени похожести тексты.

In [ ]:
# ищем в векторной базе данных фрагменты текста, которые по смыслу ближе всего к поисковому запросу
def semantic_search(query: str, n: int = 3):
    results = vectorstore.similarity_search_with_score(query, k=n)
    return results

print("Функция реализована")

Функция реализована


In [ ]:
# тестируем поиск на разных запросах
test_queries = [
    "movie plot and story",
    "acting performance actors",
    "special effects visuals"
]

for query in test_queries:
    print(f"Запрос: '{query}'")
    results = semantic_search(query, n=2)

    for i, (doc, score) in enumerate(results, 1):
        print(f"  Результат {i}: score = {score:.4f}")
        # показываем первые 150 символов, убираем переносы строк
        text_preview = doc.page_content[:150].replace('\n', ' ')
        print(f"Текст: {text_preview}...")
        print(f"Метаданные: {doc.metadata}")

Запрос: 'movie plot and story'
  Результат 1: score = 0.8086
Текст: Excellent plot within a plot within a plot. Shame about two of my film heroes having a good snog. Must be my upbringing:)<br /><br />Very well acted b...
Метаданные: {'source': 'reviews_films_imbd', 'id': 892}
  Результат 2: score = 0.8681
Текст: It isn't always easy to explain what a movie is like, but this time I think I've found it. It reminded me of two movies: Trainspotting (small time cri...
Метаданные: {'source': 'reviews_films_imbd', 'id': 250}
Запрос: 'acting performance actors'
  Результат 1: score = 1.0033
Текст: This seems like two films: one a dreary, pretentious lengthy saga about an ac-tor who is taken over by the parts he plays; the other a brilliant socia...
Метаданные: {'id': 160, 'source': 'reviews_films_imbd'}
  Результат 2: score = 1.0820
Текст: An annoying experience. Improvised dialogue, handheld cameras for no effect, directionless plot, contrived romance, ick! to the whole mess. Ron Silver...
М

# Промпт и генерация итогового ответа

In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

OPEN_ROUTER_KEY = userdata.get('OPEN_ROUTER_KEY')  # берем ключ из секретов

llm = ChatOpenAI(
    api_key=OPEN_ROUTER_KEY,
    base_url="https://openrouter.ai/api/v1",
    model="deepseek/deepseek-r1",# подключаемся к модели
    temperature=0.3,
    max_tokens=512
)
print("Модель подключена")

Модель подключена


In [ ]:
def get_rag_answer(query, n_chunks=3):
    # ищет релевантные чанки
    results = semantic_search(query, n=n_chunks)

    if not results:
        return "Не удалось найти релевантную информацию.", []

    # формирует контекст из найденных чанков
    context_parts = []
    for i, (doc, score) in enumerate(results, 1):
        context_parts.append(f"[Чанк {i}] (score: {score:.4f})\n{doc.page_content}")
    context = "\n\n---\n\n".join(context_parts)

    # промпт для LLM
    prompt = f"""You are a helpful assistant. Answer the question based ONLY on the following context.
If you cannot find the answer in the context, say "I cannot find this information in the reviews."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    # генерация ответа
    try:
        response = llm.invoke(prompt)
        answer = response.content if hasattr(response, 'content') else str(response)
        return answer, results
    except Exception as e:
        return f"Ошибка: {e}", results

print("Функция get_rag_answer определена")


Функция get_rag_answer определена


In [ ]:
# вызов функции
query = "What do reviewers say about the acting?"
answer, chunks = get_rag_answer(query, n_chunks=2)
print("Ответ:", answer)

Ответ: Reviewers have mixed opinions about the acting. Noni Hazelhurst's performance is praised as "quite good," and Angie Dickinson's portrayal of Kate is highlighted as "very well developed," with her work described as her best since *Point Blank*. However, the bulk of the cast is criticized for "wooden acting," and Colin Friels is singled out as "woefully miscast" and "ever overrated." Some reviewers also note that characters feel unrealistic, with their behaviors and dialogue conflicting with their supposed roles (e.g., "absurdly well-spoken junkies").


# Тестирование

In [65]:
# тестируем RAG-систему
questions = [
    "What do reviewers say about the acting in movies?",
    "What are the main strengths and weaknesses of movies according to reviews?",
    "Why do some reviewers criticize the plot?"
]

for q in questions:
    print("\n" + "="*60)
    print("QUESTION:", q)
    print("="*60)

    answer, chunks = get_rag_answer(q, n_chunks=3)

    print("\nANSWER:")
    print("-"*50)
    print(answer)
    print("-"*50)

    print("\nUSED CHUNKS:")
    for i, (doc, score) in enumerate(chunks, 1):
        print(f"{i}. score = {score:.4f}")
        print(f"текст: {doc.page_content}")
        print(f"id: {doc.metadata.get('id', 'unknown')}")
        print()


QUESTION: What do reviewers say about the acting in movies?

ANSWER:
--------------------------------------------------
Based solely on the provided context, reviewers have conflicting opinions about the acting in the movies discussed:

1.  **Negative Views:**  
    *   From Чанк 1: "The acting is wooden and stiff."
    *   From Чанк 3: "Where did they get these awful actors from?", "The actors in this film are appalling. Almost as bad as 'Sunset Beach.' - Extremely corny and badly performed." The reviewer specifically criticizes Amy Adams ("rigid, pathetic and badly thought out performance") and Robin Dunne ("poor").

2.  **Positive View:**  
    *   From Чанк 2: "Dickinson's character Kate is very well developed and her performance is felt throughout the entire film. The best work Angie Dickinson did since Point Blank!" This reviewer praises Angie Dickinson's acting highly.

Therefore, reviewers say the acting ranges from **appalling, wooden, stiff, corny, and poorly performed** to 

# Выводы

К сожалению, систему нельзя назвать идеальной, однако мы считаем, что задуманный семантический поиск выполняется за счёт подсоединения LLM. Мы видим, что score не так близок к 0, как нам хотелось бы, однако на это может быть несколько причин. Во-первых, выбор в пользу цельности чанков, из-за чего они получаются объемными, что влияет на их эмббединги. Во-вторых, непосредственное содержание отзывов: это неупорядоченные данные с живым языком, поэтому для машины могут возникать трудности на этапе векторизации и последующего извлечения информации из базы данных. Тем не менее, на наш взгляд, итоговые ответы и выбираемые чанки уместны, чему компенсирует LLM, понимающая контексты и учитывающая лексическую вариативность.